# Strand-Level Framework for Multi-Edge Multiplicities

This notebook explains how strand-level solutions are generated from your port-level solutions, and how to adapt that logic to heterogeneous edge multiplicities.

Target use case:
- You already generate valid port-level solutions in `TopologyMultiEdge`.
- You now want all **unique strand-level assignments** for each port-level solution.

## 1) What Changes Between Port Level and Strand Level

At port level, a solution is a multiset of paths with multiplicities:
- Example: `{(0,1):2, (2,3):1}` means path `(0,1)` appears twice and `(2,3)` once.

At strand level, each path instance gets concrete strand IDs on each edge:
- Example strand-path: `[(0, 1), (1, 0)]` means edge 0 uses strand id 1 and edge 1 uses strand id 0 for that path instance.

For each edge `e`, strand IDs are `0..T[e]-1` where `T[e]` is the edge target multiplicity.
Every strand id on an edge must be used exactly once across all instantiated path copies.

## 2) Mathematical Constraint at Strand Level

Given a fixed port-level solution, expand multiplicities into concrete path instances.
For each instance and each edge in that path, assign one strand ID.

Constraints:
1. If a path instance uses edge `e`, it consumes one unused strand id from edge `e`.
2. For each edge `e`, all ids in `0..T[e]-1` are used exactly once by the end.

This is a constrained assignment (backtracking) problem, not a path-generation problem.

In [ ]:
from collections import Counter
from itertools import product

def expand_port_solution(port_solution):
    """
    Convert a port-level solution (iterable of (path, multiplicity))
    into a flat list of path instances.
    """
    instances = []
    for path, multiplicity in port_solution:
        for _ in range(multiplicity):
            instances.append(tuple(path))
    return instances

def edge_usage_from_port_solution(port_solution, edge_ids):
    counts = Counter({e: 0 for e in edge_ids})
    for path, multiplicity in port_solution:
        for e in path:
            counts[e] += multiplicity
    return counts

def validate_port_solution_matches_targets(port_solution, edge_targets):
    edge_ids = list(edge_targets.keys())
    counts = edge_usage_from_port_solution(port_solution, edge_ids)
    return all(counts[e] == edge_targets[e] for e in edge_ids), counts

In [ ]:
def canonicalize_strand_solution(strand_solution):
    """
    Canonicalize a strand-level solution under per-edge strand relabeling.

    strand_solution: list of path instances, where each path instance is
    a list/tuple of (edge_id, strand_id).

    Why needed:
    - Many assignments differ only by renaming strand IDs on an edge
      (for example swapping ids 0 and 1 on edge 3).
    - Those should be considered equivalent.
    """
    per_edge_relabel = {}
    next_label = {}

    normalized_paths = []
    for path in strand_solution:
        normalized_path = []
        for e, s in path:
            if e not in per_edge_relabel:
                per_edge_relabel[e] = {}
                next_label[e] = 0
            if s not in per_edge_relabel[e]:
                per_edge_relabel[e][s] = next_label[e]
                next_label[e] += 1
            normalized_path.append((e, per_edge_relabel[e][s]))
        normalized_paths.append(tuple(normalized_path))

    # Sorting path instances removes ordering artifacts between identical multiplicity copies.
    return tuple(sorted(normalized_paths))

In [ ]:
def enumerate_strand_solutions(port_solution, edge_targets, break_early=None):
    """
    Enumerate unique strand-level assignments for one fixed port-level solution.

    Input:
    - port_solution: iterable of (path, multiplicity), path is tuple of edge IDs
    - edge_targets: dict {edge_id: target multiplicity}

    Output:
    - list of unique strand solutions
      each solution is tuple(sorted(path_instances)), and each path_instance is
      tuple of (edge, strand_id).
    """
    ok, counts = validate_port_solution_matches_targets(port_solution, edge_targets)
    if not ok:
        raise ValueError(
            f"Port solution does not match edge targets. counts={dict(counts)}, targets={edge_targets}"
        )

    path_instances = expand_port_solution(port_solution)
    edge_ids = sorted(edge_targets.keys())

    # Available strand IDs per edge.
    available = {e: list(range(edge_targets[e])) for e in edge_ids}

    unique_canonical = set()
    raw_solutions = []

    def backtrack(i, current_assignment):
        if break_early is not None and len(unique_canonical) >= break_early:
            return

        if i == len(path_instances):
            canonical = canonicalize_strand_solution(current_assignment)
            if canonical not in unique_canonical:
                unique_canonical.add(canonical)
                raw_solutions.append(canonical)
            return

        path = path_instances[i]
        choices = [available[e] for e in path]

        # One id choice per edge in this path instance.
        for choice in product(*choices):
            # Reserve chosen IDs
            reserved = []
            feasible = True
            for e, sid in zip(path, choice):
                if sid not in available[e]:
                    feasible = False
                    break
                available[e].remove(sid)
                reserved.append((e, sid))

            if feasible:
                current_assignment.append(tuple(zip(path, choice)))
                backtrack(i + 1, current_assignment)
                current_assignment.pop()

            # Undo reservations
            for e, sid in reserved:
                available[e].append(sid)

            # Keep deterministic behavior
            for e in set(path):
                available[e].sort()

    backtrack(0, [])
    return raw_solutions

## 3) Why This Matches Your Original `topology_og.py` Logic

The original code has three related implementations:
- `instantiate_strands`: one selected assignment index (or default deterministic one).
- `instantiate_strands_v1`: recursive full enumeration with compatibility checks.
- `instantiate_strands_v2`: stack-based DFS for speed.

The function above is conceptually closest to `instantiate_strands_v1`, but generalized to `T[e]` per edge instead of a single global `N`.

Core equivalence:
- old `available_edge_ids = {e: [0..N-1]}`
- new `available_edge_ids = {e: [0..T[e]-1]}`

Everything else (reserve ids, recurse, undo on backtrack) is the same pattern.

In [ ]:
def estimate_assignment_count(port_solution, edge_targets):
    """
    Extension of your old cardinality formula:
      prod_e (T[e]!) / prod_path (multiplicity(path)!)

    This is an upper-structure count used for intuition and sanity checks.
    """
    import math

    numerator = 1
    for e, t in edge_targets.items():
        numerator *= math.factorial(t)

    denominator = 1
    for _, multiplicity in port_solution:
        denominator *= math.factorial(multiplicity)

    return numerator // denominator

In [ ]:
# Toy example independent of lattice files
edge_targets = {0: 2, 1: 2, 2: 1}

# Port-level solution as iterable of (path, multiplicity)
# path (0,1) appears twice, path (1,2) appears once
port_solution = [
    ((0, 1), 2),
    ((1, 2), 1),
]

ok, counts = validate_port_solution_matches_targets(port_solution, edge_targets)
print("Port solution valid against targets:", ok)
print("Counts:", dict(counts))
print("Estimated assignment count:", estimate_assignment_count(port_solution, edge_targets))

strand_solutions = enumerate_strand_solutions(port_solution, edge_targets, break_early=20)
print("Unique strand solutions found (possibly truncated by break_early):", len(strand_solutions))
if strand_solutions:
    print("First strand solution:")
    for strand_path in strand_solutions[0]:
        print("  ", strand_path)

## 4) Adapter for Your Current `TopologyMultiEdge` Object

Use this helper to inspect strand solutions for one computed port-level solution in your current class.

Expected object fields (already present in your file):
- `topo.solutions` as list of frozensets of `(path, multiplicity)`
- `topo.edge_targets` as dict `{edge_id: multiplicity}`

In [ ]:
def strand_solutions_for_topology_multi_edge(topo, solution_index=0, break_early=None):
    if not topo.solutions:
        raise ValueError("No port-level solutions found. Run topo.generate_solutions() first.")

    if solution_index < 0 or solution_index >= len(topo.solutions):
        raise IndexError(f"solution_index {solution_index} out of range [0, {len(topo.solutions)-1}]")

    # topo.solutions[solution_index] is frozenset((path, multiplicity), ...)
    port_solution = sorted(list(topo.solutions[solution_index]), key=lambda x: (len(x[0]), x[0], x[1]))

    return enumerate_strand_solutions(
        port_solution=port_solution,
        edge_targets=topo.edge_targets,
        break_early=break_early,
    )

## 5) Integration Plan for Your Codebase

Add these pieces into `TopologyMultiEdge`:
1. Data fields
- `self.strand_solutions = {}` keyed by port-level solution index.
- optional `self.unique_strand_solutions = {}` if you keep both raw and canonical forms.

2. New methods
- `_expand_port_solution(solution_index)`
- `_canonicalize_strand_solution(strand_solution)`
- `generate_strand_solutions(solution_index, break_early=None)`

3. Uniqueness policy
- minimum: quotient by per-edge strand relabeling (implemented above).
- optional: further quotient by graph symmetries if needed.

4. Performance notes
- Strand enumeration can explode combinatorially.
- Keep `break_early` while debugging.
- Reorder path instances to improve pruning (longer paths first often helps).

Once this is stable, you can add your `compute_loops` stage exactly as before, now operating on heterogeneous strand IDs.